# OpenDataBattle2026 Phase2 データ加工【盆地】

#### [1]読み込み
#### [2]前処理

## 【読み込み】

### ライブラリ読み込み

In [1]:
import pandas as pd

import geopandas as gpd
from shapely.geometry import Point

from pyproj import Geod

### データ読み込み

In [2]:
# 県庁所在地一覧
df_prefectural_capitals = pd.read_csv("../data/raw_00_reference_県庁所在地一覧_20260803.csv", encoding='shift_jis')

# 日本百名山データ
df_hyakumeizan = pd.read_csv("../data/raw_03_日本百名山_20260901.csv", encoding="cp932")

In [3]:
# 各県庁所在地の庁舎の緯度経度をデータフレーム化
### ソース元：都道府県庁所在地 緯度経度データ- https://www.benricho.org/chimei/latlng_data.html
data_capital_location = [
    ["北海道", "札幌市", 43.064310, 141.346879],
    ["青森県", "青森市", 40.824589, 140.740548],
    ["岩手県", "盛岡市", 39.703526, 141.152696],
    ["宮城県", "仙台市", 38.268579, 140.872072],
    ["秋田県", "秋田市", 39.718626, 140.102381],
    ["山形県", "山形市", 38.240434, 140.363690],
    ["福島県", "福島市", 37.750029, 140.467771],
    ["茨城県", "水戸市", 36.341737, 140.446824],
    ["栃木県", "宇都宮市", 36.565912, 139.883592],
    ["群馬県", "前橋市", 36.390688, 139.060453],
    ["埼玉県", "さいたま市", 35.857033, 139.649012],
    ["千葉県", "千葉市", 35.604560, 140.123154],
    ["東京都", "新宿区", 35.689501, 139.691722],
    ["神奈川県", "横浜市", 35.447734, 139.642537],
    ["新潟県", "新潟市", 37.902451, 139.023245],
    ["富山県", "富山市", 36.695265, 137.211305],
    ["石川県", "金沢市", 36.594606, 136.625669],
    ["福井県", "福井市", 36.065209, 136.221720],
    ["山梨県", "甲府市", 35.664108, 138.568455],
    ["長野県", "長野市", 36.651306, 138.180904],
    ["岐阜県", "岐阜市", 35.391174, 136.723657],
    ["静岡県", "静岡市", 34.976944, 138.383056],
    ["愛知県", "名古屋市", 35.180209, 136.906582],
    ["三重県", "津市", 34.730278, 136.508611],
    ["滋賀県", "大津市", 35.004513, 135.868568],
    ["京都府", "京都市", 35.021242, 135.755613],
    ["大阪府", "大阪市", 34.686344, 135.520037],
    ["兵庫県", "神戸市", 34.691257, 135.183078],
    ["奈良県", "奈良市", 34.685274, 135.832861],
    ["和歌山県", "和歌山市", 34.226111, 135.167500],
    ["鳥取県", "鳥取市", 35.503449, 134.238261],
    ["島根県", "松江市", 35.472293, 133.050520],
    ["岡山県", "岡山市", 34.661739, 133.935032],
    ["広島県", "広島市", 34.396558, 132.459646],
    ["山口県", "山口市", 34.186041, 131.470654],
    ["徳島県", "徳島市", 34.065761, 134.559286],
    ["香川県", "高松市", 34.340112, 134.043291],
    ["愛媛県", "松山市", 33.841642, 132.765682],
    ["高知県", "高知市", 33.559722, 133.531111],
    ["福岡県", "福岡市", 33.606389, 130.417968],
    ["佐賀県", "佐賀市", 33.249351, 130.298792],
    ["長崎県", "長崎市", 32.750040, 129.867251],
    ["熊本県", "熊本市", 32.789800, 130.741584],
    ["大分県", "大分市", 33.238130, 131.612645],
    ["宮崎県", "宮崎市", 31.911034, 131.423887],
    ["鹿児島県", "鹿児島市", 31.560171, 130.558025],
    ["沖縄県", "那覇市", 26.212445, 127.680922],
]
df_capital_location = pd.DataFrame(
    data_capital_location,
    columns=["都道府県", "市区", "緯度", "経度"]
)

## 【データ確認】

### 県庁所在地一覧データ

In [4]:
# 末尾３件確認
df_prefectural_capitals.tail(3)

,コード,都道府県,県庁所在地,ローマ字
44,45,宮崎県,宮崎市,miyazaki
45,46,鹿児島県,鹿児島市,kagoshima
46,47,沖縄県,那覇市,naha


In [5]:
# 概要
df_prefectural_capitals.info()

<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   コード     47 non-null     int64
 1   都道府県    47 non-null     str  
 2   県庁所在地   47 non-null     str  
 3   ローマ字    47 non-null     str  
dtypes: int64(1), str(3)
memory usage: 1.6 KB


### 日本百名山データ

In [6]:
# 末尾３件確認
df_hyakumeizan.head(3)

,No,山名,北緯,東経,よみがな,標高（m）,所在地,地域名,備考
0,1,利尻山（利尻富士）,45.178611,141.241944,りしりざん（りしりふじ）,1721,北海道,北海道,NaN
1,2,羅臼岳,44.075833,145.122222,らうすだけ,1661,北海道,北海道,H26年標高変更
2,3,斜里岳,43.765555,144.717778,しゃりだけ,1547,北海道,北海道,NaN


In [7]:
# 概要
df_hyakumeizan.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   No      100 non-null    int64  
 1   山名      100 non-null    str    
 2   北緯      100 non-null    float64
 3   東経      100 non-null    float64
 4   よみがな    100 non-null    str    
 5   標高（m）   100 non-null    int64  
 6   所在地     100 non-null    str    
 7   地域名     100 non-null    str    
 8   備考      12 non-null     str    
dtypes: float64(2), int64(2), str(5)
memory usage: 7.2 KB


### 庁舎の緯度経度データ

In [8]:
df_capital_location.head(3)

,都道府県,市区,緯度,経度
0,北海道,札幌市,43.064310,141.346879
1,青森県,青森市,40.824589,140.740548
2,岩手県,盛岡市,39.703526,141.152696


In [9]:
df_capital_location.info()

<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   都道府県    47 non-null     str    
 1   市区      47 non-null     str    
 2   緯度      47 non-null     float64
 3   経度      47 non-null     float64
dtypes: float64(2), str(2)
memory usage: 1.6 KB


## 【前処理】

### 県庁所在地データ

In [10]:
### 東京都の県庁所在地が「東京都」になっているので「東京」に変更
df_prefectural_capitals["県庁所在地"] = df_prefectural_capitals["県庁所在地"].replace(
    "東京都", "東京"
)
df_prefectural_capitals["県庁所在地"].unique()

<StringArray>
[  '札幌市',   '青森市',   '盛岡市',   '仙台市',   '秋田市',   '山形市',   '福島市',   '水戸市',
  '宇都宮市',   '前橋市', 'さいたま市',   '千葉市',    '東京',   '横浜市',   '新潟市',   '富山市',
   '金沢市',   '福井市',   '甲府市',   '長野市',   '岐阜市',   '静岡市',  '名古屋市',    '津市',
   '大津市',   '京都市',   '大阪市',   '神戸市',   '奈良市',  '和歌山市',   '鳥取市',   '松江市',
   '岡山市',   '広島市',   '山口市',   '徳島市',   '高松市',   '松山市',   '高知市',   '福岡市',
   '佐賀市',   '長崎市',   '熊本市',   '大分市',   '宮崎市',  '鹿児島市',   '那覇市']
Length: 47, dtype: str

In [11]:
# 結合キー用に都市名から"市"を抜いた列を作成
### 末尾についた"市"だけ削除removesuffix()
df_prefectural_capitals["都市"] = (
    df_prefectural_capitals["県庁所在地"]
    .str.removesuffix("市")
)
df_prefectural_capitals.head()

,コード,都道府県,県庁所在地,ローマ字,都市
0,1,北海道,札幌市,sapporo,札幌
1,2,青森県,青森市,aomori,青森
2,3,岩手県,盛岡市,morioka,盛岡
3,4,宮城県,仙台市,sendai,仙台
4,5,秋田県,秋田市,akita,秋田


### 庁舎の緯度経度と県庁所在地データを結合

In [12]:
df_capital_location = df_capital_location.merge(
    df_prefectural_capitals[["都道府県", "都市"]],
    on="都道府県",
    how="left"
)
df_capital_location.head(3)

,都道府県,市区,緯度,経度,都市
0,北海道,札幌市,43.064310,141.346879,札幌
1,青森県,青森市,40.824589,140.740548,青森
2,岩手県,盛岡市,39.703526,141.152696,盛岡


In [13]:
# GIS用にPoint化しておく
gdf_capital_points = gpd.GeoDataFrame(
    df_capital_location.copy(),
    geometry=[
        Point(lon, lat)
        for lon, lat in zip(
            df_capital_location["経度"],
            df_capital_location["緯度"]
        )
    ],
    crs="EPSG:4326"
)

In [14]:
print(gdf_capital_points.shape)
print(gdf_capital_points.crs)
print(gdf_capital_points.head())

(47, 6)
EPSG:4326
  都道府県   市区         緯度          経度  都市                    geometry
0  北海道  札幌市  43.064310  141.346879  札幌  POINT (141.34688 43.06431)
1  青森県  青森市  40.824589  140.740548  青森  POINT (140.74055 40.82459)
2  岩手県  盛岡市  39.703526  141.152696  盛岡   POINT (141.1527 39.70353)
3  宮城県  仙台市  38.268579  140.872072  仙台  POINT (140.87207 38.26858)
4  秋田県  秋田市  39.718626  140.102381  秋田  POINT (140.10238 39.71863)


### 百名山データ

In [15]:
# 百名山の北緯東経をPoint化
gdf_hyakumeizan = gpd.GeoDataFrame(
    df_hyakumeizan.copy(),
    geometry=[
        Point(lon, lat)
        for lon, lat in zip(
            df_hyakumeizan["東経"],
            df_hyakumeizan["北緯"]
        )
    ],
    crs="EPSG:4326"
)

## 【各県庁所在地の庁舎の最寄り百名山までの距離】

In [16]:
# 1. 都市と百名山の全組み合わせ
df_capital = gdf_capital_points[
    ["都市", "geometry"]
].rename(
    columns={"geometry": "geometry_city"}
)

df_mountain = gdf_hyakumeizan[
    ["山名", "geometry"]
].rename(
    columns={"geometry": "geometry_mountain"}
)

df_pairs = (
    df_capital.assign(key=1)
    .merge(
        df_mountain.assign(key=1),
        on="key"
    )
    .drop(columns="key")
)

print(df_pairs.shape)

(4700, 4)


In [17]:
# 2. 測地線距離を計算
geod = Geod(ellps="WGS84")

_, _, distance = geod.inv(
    df_pairs["geometry_city"].x,
    df_pairs["geometry_city"].y,
    df_pairs["geometry_mountain"].x,
    df_pairs["geometry_mountain"].y
)

df_pairs["距離_km"] = distance / 1000

In [18]:
# 3. 各都市の最寄り百名山を取得
idx = df_pairs.groupby("都市")["距離_km"].idxmin()

df_nearest_mountain = (
    df_pairs.loc[
        idx,
        ["都市", "山名", "距離_km"]
    ]
    .sort_values("都市")
    .reset_index(drop=True)
)

### カラム名整理
df_nearest_mountain_feature = (
    df_nearest_mountain[
        ["都市", "山名", "距離_km"]
    ]
    .rename(columns={
        "山名": "最寄り百名山",
        "距離_km": "最寄り百名山距離_km"
    })
)

df_nearest_mountain_feature

,都市,最寄り百名山,最寄り百名山距離_km
0,さいたま,筑波山,58.055862
1,京都,伊吹山,73.795862
2,仙台,蔵王山（熊野岳）,40.305631
3,佐賀,阿蘇山（高岳）,85.382125
4,前橋,赤城山（黒?山）,22.270392
5,千葉,筑波山,68.889315
6,名古屋,伊吹山,52.578431
7,和歌山,大峰山（八経ヶ岳）,68.452233
8,大分,くじゅう連山（久住山）,38.755282
9,大津,伊吹山,67.083685


## 【各県庁所在地の庁舎から半径50km以内にある百名山の数】

In [19]:
# 各都市から50km以内にある日本百名山の数
df_mountains_within_50km = (
    df_pairs[df_pairs["距離_km"] <= 50]
    .groupby("都市")
    .size()
    .reset_index(name="百名山_50km以内")
)

# 47都市すべてを対象にし、該当なしは0
df_mountains_within_50km = (
    df_prefectural_capitals[["都市"]]
    .merge(
        df_mountains_within_50km,
        on="都市",
        how="left"
    )
    .fillna({"百名山_50km以内": 0})
)

# 整数型にする
df_mountains_within_50km["百名山_50km以内"] = (
    df_mountains_within_50km["百名山_50km以内"]
    .astype(int)
)

df_mountains_within_50km

,都市,百名山_50km以内
0,札幌,0
1,青森,2
2,盛岡,3
3,仙台,1
4,秋田,0
5,山形,3
6,福島,4
7,水戸,1
8,宇都宮,2
9,前橋,5


In [20]:
df_mountains_within_50km_detail = (
    df_pairs[df_pairs["距離_km"] <= 50]
    [["都市", "山名", "距離_km"]]
    .sort_values(["都市", "距離_km"])
    .reset_index(drop=True)
)

In [21]:
# 県庁所在地データの都市順を取得
city_order = df_prefectural_capitals["都市"].tolist()

# 都市を県庁所在地データの順序で並べる
df_mountains_within_50km_detail["都市"] = pd.Categorical(
    df_mountains_within_50km_detail["都市"],
    categories=city_order,
    ordered=True
)

# 都市順 → 距離順でソート
df_mountains_within_50km_detail = (
    df_mountains_within_50km_detail
    .sort_values(["都市", "距離_km"])
    .reset_index(drop=True)
)

df_mountains_within_50km_detail

,都市,山名,距離_km
0,青森,八甲田山（大岳）,21.722137
1,青森,岩木山,41.432903
2,盛岡,岩手山,21.029379
3,盛岡,早池峰山,33.023495
4,盛岡,八幡平,38.078929
...,...,...,...
64,松山,石鎚山（天狗岳）,33.366803
65,高知,石鎚山（天狗岳）,44.966227
66,熊本,阿蘇山（高岳）,35.507427
67,大分,くじゅう連山（久住山）,38.755282


In [28]:
# 都市ごとに連番を振る
df_mountains_within_50km_detail["連番"] = df_mountains_within_50km_detail.groupby("都市").cumcount() + 1
df_mountains_within_50km_detail.head(5)

,都市,山名,距離_km,連番
0,青森,八甲田山（大岳）,21.722137,1
1,青森,岩木山,41.432903,2
2,盛岡,岩手山,21.029379,1
3,盛岡,早池峰山,33.023495,2
4,盛岡,八幡平,38.078929,3


## 【作成した特徴量をマージする】

In [22]:
df_analysis_mountain = (
    df_prefectural_capitals
    .merge(
        df_nearest_mountain_feature,
        on="都市",
        how="left"
    )
    .merge(
        df_mountains_within_50km,
        on="都市",
        how="left"
    )
)

In [23]:
df_analysis_mountain.head(2)

,コード,都道府県,県庁所在地,ローマ字,都市,最寄り百名山,最寄り百名山距離_km,百名山_50km以内
0,1,北海道,札幌市,sapporo,札幌,後方羊蹄山（羊蹄山）,51.057849,0
1,2,青森県,青森市,aomori,青森,八甲田山（大岳）,21.722137,2


In [29]:
# csv出力
### 各庁舎の緯度経度データ
df_capital_location.to_csv("../data/data_capital_location.csv")
### 各庁舎から半径50km以内にある百名山リスト
df_mountains_within_50km_detail.to_csv("../data/data_bonchi_mountains_within_50km_detail.csv")
### 集計結果
df_analysis_mountain.to_csv("../data/df_analysis_bonchi_mountain.csv")